In [1]:
from pathlib import Path

# The notebooks are stored in <package>/notebooks.
notebooks_dir = Path.cwd()
if notebooks_dir.name != "notebooks":
    candidate = notebooks_dir / "notebooks"
    if candidate.exists():
        notebooks_dir = candidate

project_root = notebooks_dir.parent
nhanes_dir = project_root / "src" / "nhanes"
jordan_dir = project_root / "src" / "jordan_external_validation"
expected_nhanes = project_root / "outputs" / "nhanes"
expected_jordan = project_root / "outputs" / "jordan"

assert (project_root / "README.md").exists(), project_root
print("Project root:", project_root)

Project root: <project_root>


In [2]:
import json
import pandas as pd

live = nhanes_dir / "outputs"
results = live if (live / "common_logistic_results.csv").exists() else expected_nhanes
print("Reading results from:", results)

Reading results from: <project_root>\outputs\nhanes


In [3]:
cohorts = pd.read_csv(results / "cohort_and_split_composition.csv")
assignment = pd.read_csv(results / "psu_assignment.csv")
display(cohorts)

assert assignment.groupby("psu_group")["split"].nunique().max() == 1
display(assignment.groupby("split").agg(psus=("psu_group", "nunique"), strata=("SDMVSTRA", "nunique")))
print("PASS: no PSU overlap across partitions.")

,panel,split,n,positive,nonpositive,borderline,psus,strata,weight_total,weighted_positive_proportion
0,mec_nonfasting,all,2589,459,2130,0,30,15,1.158530e+08,0.129290
1,mec_nonfasting,test,515,93,422,0,6,3,2.243015e+07,0.155660
2,mec_nonfasting,train,1551,271,1280,0,18,9,7.019660e+07,0.119338
3,mec_nonfasting,validation,523,95,428,0,6,3,2.322626e+07,0.133901
4,fasting_with_glucose,all,1096,213,883,0,30,15,1.171780e+08,0.126867
5,fasting_with_glucose,test,215,44,171,0,6,3,2.252354e+07,0.176346
6,fasting_with_glucose,train,654,124,530,0,18,9,7.066564e+07,0.102207
7,fasting_with_glucose,validation,227,45,182,0,6,3,2.398882e+07,0.153055
8,fasting_without_glucose,all,1096,213,883,0,30,15,1.171780e+08,0.126867
9,fasting_without_glucose,test,215,44,171,0,6,3,2.252354e+07,0.176346


,psus,strata
split,,
test,6,3
train,18,9
validation,6,3


PASS: no PSU overlap across partitions.


In [4]:
models = pd.read_csv(results / "common_logistic_results.csv")
paired = pd.read_csv(results / "paired_logistic_auc_differences.csv")

display(models[["panel", "config", "auc", "brier", "calibration_in_the_large", "calibration_slope"]].round(4))
display(paired.round(4))

primary = models.query("panel == 'mec_nonfasting' and config == 'BOTH'").iloc[0]
assert round(primary.auc, 4) == 0.8863
assert round(primary.brier, 4) == 0.0866
print("PASS: primary headline results match the expected analysis.")

,panel,config,auc,brier,calibration_in_the_large,calibration_slope
0,mec_nonfasting,BASE,0.8884,0.0893,0.1736,0.8796
1,mec_nonfasting,BMI,0.8833,0.0897,0.1997,0.8704
2,mec_nonfasting,BOTH,0.8863,0.0866,0.2043,0.8718
3,mec_nonfasting,WHR,0.8897,0.0857,0.1932,0.8881
4,fasting_with_glucose,BASE,0.9032,0.0742,0.0206,0.6561
5,fasting_with_glucose,BMI,0.9015,0.0751,0.1709,0.6360
6,fasting_with_glucose,BOTH,0.9047,0.0715,0.1178,0.7180
7,fasting_with_glucose,WHR,0.9067,0.0714,0.0660,0.7317
8,fasting_without_glucose,BASE,0.8255,0.1006,0.1693,0.6645
9,fasting_without_glucose,BMI,0.8459,0.0998,0.2846,0.6064


,panel,comparison,delta_auc,lower_psu,upper_psu,replicates
0,mec_nonfasting,BMI minus BASE,-0.0050,-0.0165,0.0080,1000
1,mec_nonfasting,WHR minus BASE,0.0013,-0.0074,0.0092,1000
2,mec_nonfasting,BOTH minus BASE,-0.0021,-0.0112,0.0041,1000
3,mec_nonfasting,WHR minus BMI,0.0063,-0.0034,0.0149,1000
4,mec_nonfasting,BOTH minus BMI,0.0029,-0.0039,0.0107,1000
5,fasting_with_glucose,BMI minus BASE,-0.0016,-0.0124,0.0064,1000
6,fasting_with_glucose,WHR minus BASE,0.0035,-0.0087,0.0215,1000
7,fasting_with_glucose,BOTH minus BASE,0.0015,-0.0046,0.0181,1000
8,fasting_with_glucose,WHR minus BMI,0.0051,-0.0087,0.0260,1000
9,fasting_with_glucose,BOTH minus BMI,0.0031,-0.0046,0.0221,1000


PASS: primary headline results match the expected analysis.
